# **Корреляционный анализ данных (Correlation Analysis)**

* __Цель корреляции:__ Изучить взаимосвязи между признаками внутри выделенных сегментов пациентов с помощью корреляционного анализа, научиться интерпретировать коэффициенты корреляции и выявлять мультиколлинеарность.
* __Задачи корреляции:__ Применить статистические методы для выявления силы и направления связей между переменными, оценить их значимость (p-value) и сравнить структуру корреляций в разных кластерах.
* __Алгоритм использования:__
  1. __Общая корреляционная матрица (Пирсон и Спирмен)__
  2. __Анализ статистической значимости корреляций (p-value)__
  3. __Проверка данных на мультиколлинеарность (VIF)__
  4. __Внутрикластерный корреляционный анализ (K-Means, DBSCAN, Иерархический)__

---

## **1. Общий корреляционный анализ**

* __Цель:__ Выявить и оценить базовые статистические взаимосвязи между всеми числовыми медицинскими признаками на уровне всей выборки пациентов (до разделения на специфические сегменты).
* __Задачи:__
  1. Рассчитать и визуально сравнить корреляционные матрицы Пирсона и Спирмена для определения природы связей (линейные или монотонные).
  2. Построить наглядные тепловые карты (Heatmaps) с отсечением дублирующейся информации для быстрого поиска наиболее сильных зависимостей.
  3. Вычислить статистическую значимость (p-value) каждого коэффициента для фильтрации случайного информационного шума и формирования достоверной матрицы статистически значимых корреляций.

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats

from scripts.logger import setup_logger

# Инициализация логгера
logger = setup_logger(__name__)

# Настройка визуализации
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 8)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

In [ ]:
# --- 1. ЗАГРУЗКА ДАННЫХ И ПЕРВИЧНЫЙ АНАЛИЗ ---

try:
    data = pd.read_csv("../data/processed/heart_disease_uci_clustered.csv")
    display(Markdown("#### **Содержание набора данных с метками кластеров**"))
    display(data.head())
    display(
        Markdown(
            f"Размерность данных: **{data.shape[0]} строк и {data.shape[1]} столбцов.**"
        )
    )
except FileNotFoundError as e:
    logger.error(
        "Файл `../data/processed/heart_disease_uci_clustered.csv` не найден. Убедитесь, что пайплайн кластеризации был выполнен."
    )
    raise e

In [ ]:
# --- 2. ПОДГОТОВКА ДАННЫХ ДЛЯ АНАЛИЗА ---

display(Markdown("#### **Определение признаков для анализа**"))

# Определение признаков для анализа
cluster_cols = ["Cluster_KMeans", "Cluster_DBSCAN", "Cluster_Hierarchical"]
columns_to_exclude = set(cluster_cols + ["id", "num"])
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

# Фильтрация признаков
feature_cols = [col for col in numeric_cols if col not in columns_to_exclude]

display(
    Markdown(
        f"* Отобрано **{len(feature_cols)}** числовых признаков для корреляционного анализа.\n\n"
        f"* **Признаки:** `{', '.join(feature_cols)}`."
    )
)

# Создание словарей с датасетами для внутрисегментного анализа
cluster_datasets = {}

for col in cluster_cols:
    algo_name = col.split("_")[1]

    cluster_datasets[algo_name] = {
        c_id: data[data[col] == c_id][feature_cols].copy()
        for c_id in sorted(data[col].dropna().unique())
    }

display(Markdown("#### **Структура разбиения данных по кластерам**"))
display(
    Markdown(
        f"* **K-Means:** {[int(k) for k in cluster_datasets['KMeans'].keys()]}\n\n"
        f"* **DBSCAN:** {[int(k) for k in cluster_datasets['DBSCAN'].keys()]} (где -1 - это шум)\n\n"
        f"* **Иерархическая:** {[int(k) for k in cluster_datasets['Hierarchical'].keys()]}"
    )
)

# Подготовка общего набора данных для анализа
display(Markdown("#### **Общий набор данных для анализа**"))
data_features = data[feature_cols].copy()
display(data_features.head())

In [ ]:
# --- 3. ОБЩАЯ КОРРЕЛЯЦИОННАЯ МАТРИЦА ---

display(Markdown("#### **Общая корреляционная матрица (Пирсон и Спирмен)**"))

corr_pearson = data_features.corr(method="pearson")
corr_spearman = data_features.corr(method="spearman")

display(Markdown("* **Корреляционная матрица Пирсона (линейные связи):**"))
display(
    corr_pearson.style.background_gradient(cmap="coolwarm", axis=None).format(
        precision=3
    )
)

display(
    Markdown("* **Корреляционная матрица Спирмена (монотонные/нелинейные связи):**")
)
display(
    corr_spearman.style.background_gradient(cmap="coolwarm", axis=None).format(
        precision=3
    )
)

#### **Сравнительный анализ корреляционных матриц (Пирсон vs. Спирмен)**

* **Эмпирические наблюдения:** Сопоставление результатов вычисления коэффициентов линейной (Пирсон) и ранговой (Спирмен) корреляции демонстрирует высокую степень их согласованности. Абсолютная разность (дельта) значений для идентичных пар признаков минимальна и не превышает $0.02 - 0.04$. В частности, оценка взаимосвязи между возрастом (`age`) и максимальной ЧСС (`thalch`) по критерию Пирсона составляет $-0.351$, а по критерию Спирмена — $-0.329$.
* **Аналитическое заключение:** Подобная конвергенция (схождение) метрик статистически подтверждает **преимущественно линейную природу** взаимосвязей в исследуемом наборе данных. Кроме того, это свидетельствует об отсутствии значимых монотонных отклонений и экстремальных выбросов, к которым традиционно чувствителен линейный критерий. На основании этого, использование коэффициента корреляции Пирсона в качестве фундаментальной метрики для последующих этапов анализа является математически обоснованным.

In [ ]:
# --- 4. ВИЗУАЛИЗАЦИЯ КОРРЕЛЯЦИЙ ---

display(Markdown("#### **Визуализация корреляционных матриц (Пирсон и Спирмен)**"))

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Маски для скрытия верхнего треугольника
mask_pearson = np.triu(np.ones_like(corr_pearson, dtype=bool))
mask_spearman = np.triu(np.ones_like(corr_spearman, dtype=bool))

cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Визуализация корреляционной матрицы Пирсона с помощью тепловой карты (Рисунок 1)
sns.heatmap(
    corr_pearson,
    mask=mask_pearson,
    annot=True,
    fmt=".2f",
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=axes[0],
)
axes[0].set_title("Рисунок 1: Корреляция Пирсона (Линейная)", pad=15)
axes[0].grid(False)

# Визуализация корреляционной матрицы Спирмена с помощью тепловой карты (Рисунок 2)
sns.heatmap(
    corr_spearman,
    mask=mask_spearman,
    annot=True,
    fmt=".2f",
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=axes[1],
)
axes[1].set_title("Рисунок 2: Корреляция Спирмена (Ранговая / Монотонная)", pad=15)
axes[1].grid(False)

plt.tight_layout()
plt.show()

#### **Cтатистическая интерпретация выявленных паттернов**

* **Выраженная обратная корреляция: `age` (Возраст) и `thalch` (Максимальная ЧСС): $r = -0.35$**
  * *Клиническое обоснование:* Данная зависимость отражает фундаментальную физиологическую закономерность: с увеличением возраста максимальная частота сердечных сокращений, достигаемая на пике физической нагрузки, имеет естественную тенденцию к снижению. 
* **Умеренная прямая корреляция: `age` (Возраст) и `trestbps` (Артериальное давление в покое): $r = 0.24$**
  * *Клиническое обоснование:* Выявленная взаимосвязь объясняется возрастным снижением эластичности сосудистой стенки (преимущественно вследствие атеросклеротических процессов), что приводит к ожидаемому компенсаторному повышению базового артериального давления в старших возрастных когортах.
* **Умеренная прямая корреляция: `age` (Возраст) и `oldpeak` (Депрессия ST-сегмента): $r = 0.23$**
  * *Клиническое обоснование:* Статистически подтверждается, что с увеличением возраста возрастает вероятность фиксации ишемических изменений на ЭКГ (где депрессия ST является классическим маркером гипоксии миокарда) в ответ на физическую нагрузку.
* **Аналитическое наблюдение: Статистическая независимость признака `chol` (Уровень холестерина)**
  * *Интерпретация:* В масштабах всей выборки уровень холестерина демонстрирует отсутствие значимой линейной зависимости от остальных клинических показателей (коэффициенты корреляции стремятся к нулю). Отсутствие коллинеарности характеризует этот параметр как ортогональный (независимый) и высокоинформативный предиктор для построения будущих многомерных алгоритмов машинного обучения.

In [ ]:
# --- 5. АНАЛИЗ ЗНАЧИМОСТИ КОРРЕЛЯЦИЙ (P-VALUE) ---

display(Markdown("#### **Анализ статистической значимости корреляций**"))

# Уровень значимости для тестирования гипотезы о нулевой корреляции
ALPHA_LEVEL = 0.05

# Векторизованное вычисление p-value
p_values = data_features.corr(method=lambda x, y: stats.pearsonr(x, y)[1])

# Заполнение диагональных элементов NaN безопасным сопособом через маску
diag_mask = np.eye(len(p_values), dtype=bool)
p_values = p_values.mask(diag_mask, np.nan)


# Функция для цветового форматирования p-value
def highlight_pvals(val):
    if pd.isna(val):
        return "color: gray"  # Диагональ
    elif val < ALPHA_LEVEL:
        return "color: green; font-weight: bold"  # Статистически значимые корреляции
    else:
        return "color: red"  # Случайные корреляции


display(Markdown(f"* **Матрица p-value (Уровень значимости α = {ALPHA_LEVEL})**"))
display(
    Markdown(
        f"*Зеленым цветом выделены статистически значимые значения (p < {ALPHA_LEVEL}), красным — случайные.*"
    )
)
display(p_values.style.map(highlight_pvals).format(precision=4, na_rep="—"))

# --- ФОРМИРОВАНИЕ МАТРИЦЫ ЗНАЧИМЫХ КОРРЕЛЯЦИЙ ---

significant_corr = corr_pearson.copy()
significant_corr = significant_corr.mask((p_values >= ALPHA_LEVEL) | (p_values.isna()))

display(Markdown("* **Матрица статистически значимых корреляций**"))
display(
    significant_corr.style.background_gradient(
        cmap="coolwarm", axis=None, vmin=-1, vmax=1
    )
    .highlight_null(color="#f4f4f4")
    .format(precision=3, na_rep="-")
)

#### **Оценка статистической значимости корреляционных взаимосвязей**

На данном этапе исследования была проведена статистическая проверка гипотез для оценки достоверности выявленных связей.
Нулевая гипотеза ($H_0$): *«Истинный коэффициент корреляции между признаками в генеральной совокупности равен нулю (связи нет)»*. Критический уровень статистической значимости (вероятность ошибки первого рода) был зафиксирован на стандартизированной отметке $\alpha = 0.05$.

##### **Ключевые результаты оценки (на основе расчета p-value):**

1. **Доказательство достоверности выраженных зависимостей:** Эмпирические паттерны, продемонстрировавшие высокие коэффициенты на этапе первичного анализа (в частности, обратная связь между возрастом `age` и максимальной ЧСС `thalch`, а также прямая связь между возрастом `age` и артериальным давлением `trestbps`), характеризовались значениями $p\text{-value} < 0.0001$. Это свидетельствует о том, что вероятность случайного возникновения подобных взаимосвязей в выборке пренебрежимо мала, что позволяет уверенно отвергнуть нулевую гипотезу ($H_0$) в отношении данных пар признаков.

2. **Нивелирование статистического шума (исключение ложноположительных связей):** Ряд слабовыраженных коэффициентов не преодолел заданный порог статистической значимости. Показательным примером является взаимосвязь между уровнем сывороточного холестерина (`chol`) и депрессией ST-сегмента (`oldpeak`), где расчетное значение $p\text{-value}$ превысило $0.05$. В контексте математической статистики подобные связи интерпретируются как случайные выборочные флуктуации и методически обоснованно исключаются из дальнейшего анализа.

3. **Аналитическая ценность результирующей матрицы:** Сформированная итоговая матрица значимых корреляций представляет собой отфильтрованный массив данных, очищенный от стохастического (случайного) шума. Она агрегирует исключительно робастные взаимосвязи, которые могут служить надежным и математически обоснованным фундаментом для обучения предиктивных алгоритмов машинного обучения, а также для формулирования объективных клинических заключений.